In [13]:
import requests
from pathlib import Path
import pandas as pd

from geopy.geocoders import Nominatim

import json

dir = Path('C:/Users/Marcus/Documents/DSAI/Azure_WBS/')
measures_to_ignore = ["timestamp", "source_id", "condition", "precipitation_probability",
                      "precipitation_probability_6h", "icon", "fallback_source_ids"]

datelist = []

for i in ["flow/", "weight/", "humidity/", "temperature/"]:
    parquet_files = list(dir.joinpath('silver').joinpath(i).glob('*.parquet'))

    if not parquet_files:
        raise FileNotFoundError(f'No parquet files found in {dir}/{i}')

    newest_file = max(parquet_files, key=lambda p: p.stat().st_mtime)

    print(f"Reading {i}....", end="")
    df = pd.read_parquet(newest_file)
    mindate = df["timestamp"].min()
    maxdate = df["timestamp"].max() 
    print(f"\tMin:{mindate}, Max::{maxdate}")
    datelist.append(mindate)
    datelist.append(maxdate)

mindate=min(datelist)
maxdate=max(datelist)
print(f"Overall: \tMin:{mindate}, Max::{maxdate}")


geolocator = Nominatim(user_agent="geoapi")
location_name = newest_file.stem.split('__')[0]
location = geolocator.geocode(location_name)

#todo: add dictionary for location not found from name
print(f"Found {location_name} at {location.latitude}, {location.longitude}")

url = "https://api.brightsky.dev/weather"
headers = {"Accept": "application/json"}
params = {"lat": location.latitude, "lon": location.longitude, "date": mindate, "last_date": maxdate}

response = requests.get(url, headers=headers, params=params)

# Check if the request was successful
if response.status_code == 200:
    weather_data = response.json()  # Convert response to JSON format
    archive_dir = dir / 'bronze' / 'archive'
    archive_dir.mkdir(parents=True, exist_ok=True)
    archive_path = archive_dir / f"{location_name}__{mindate.strftime('%Y-%m-%dT%H-%M-%S')}.json"
    with open(archive_path, 'w', encoding='utf-8') as f:
        json.dump(weather_data, f, indent=4)

    sources_df = pd.DataFrame(weather_data["sources"])

    #weather_df = pd.DataFrame(weather_data.get('weather', []))


    for time in weather_data["weather"]:
        temp_dict = {}
        for measure in time.keys():
            if measure in measures_to_ignore:
                continue
            if measure in time.get("fallback_source_ids",{}):
                temp_dict[f"{measure}_source_distance"] = sources_df.loc[sources_df["id"] == time.get("fallback_source_ids",{}).get(measure), "distance"].values[0]
            else:
                temp_dict[f"{measure}_source_distance"] = sources_df.loc[sources_df["id"] == time["source_id"], "distance"].values[0]
        time.update(temp_dict)


    weather_df = (
        pd.DataFrame(weather_data["weather"])
        .drop(["source_id", "visibility", "condition", "icon", "precipitation_probability", "precipitation_probability_6h", "fallback_source_ids"], axis=1)
    )


    #weather_df = weather_df.rename({"lat": "station_lat", "lon": "station_lon", "height": "station_elevation"}, axis=1)

    weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])

    weather_dir = dir / 'silver' / 'weather'
    weather_dir.mkdir(parents=True, exist_ok=True)
    weather_df.to_parquet(weather_dir / f'{location_name}__{pd.Timestamp.now().strftime("%Y-%m-%dT%H-%M-%S")}.parquet', index=False)

else:
    print(f"Error: Unable to retrieve data ({response.status_code})")




Reading flow/....	Min:2017-01-01 13:15:00+00:00, Max::2019-05-31 12:15:00+00:00
Reading weight/....	Min:2017-01-01 12:00:00+00:00, Max::2019-05-31 12:00:00+00:00
Reading humidity/....	Min:2017-01-01 12:00:00+00:00, Max::2019-05-31 12:00:00+00:00
Reading temperature/....	Min:2017-01-01 13:10:00+00:00, Max::2019-05-31 12:15:00+00:00
Overall: 	Min:2017-01-01 12:00:00+00:00, Max::2019-05-31 12:15:00+00:00
Found schwartau at 54.0099465, 10.6754006


Columns with missing values: ['pressure_msl', 'sunshine', 'temperature', 'wind_direction', 'wind_speed', 'cloud_cover', 'dew_point', 'relative_humidity', 'visibility', 'wind_gust_direction', 'wind_gust_speed', 'condition', 'precipitation_probability', 'precipitation_probability_6h', 'solar', 'fallback_source_ids']
precipitation_probability       21119
precipitation_probability_6h    21119
sunshine                         5423
solar                            5205
visibility                       4744
cloud_cover                      4737
pressure_msl                     4736
temperature                      4736
wind_direction                   4736
wind_speed                       4736
dew_point                        4736
relative_humidity                4736
wind_gust_direction              4736
wind_gust_speed                  4736
condition                         510
fallback_source_ids               219
dtype: int64


In [ ]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="geoapi")
location = geolocator.geocode("schwartau")

print(location.latitude, location.longitude)


54.0099465 10.6754006
